In [1]:
import os
import glob
from typing import List, Dict, Any, Set, Tuple

import numpy as np
import pandas as pd
import redis
import langchain
import time
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

from rank_bm25 import BM25Okapi

from langchain_redis import RedisCache


# ----------------------------
# Load .env
# ----------------------------
load_dotenv()

# ----------------------------
# Config
# ----------------------------
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
REDIS_CACHE_TTL_SECONDS = int(os.getenv("REDIS_CACHE_TTL_SECONDS", "3600"))

DATA_DIR = os.getenv("DATA_DIR", "./pdf_data")

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-large")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "900"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "150"))
TOP_K = int(os.getenv("TOP_K", "4"))

# Hybrid knobs
HYBRID_DENSE_K = int(os.getenv("HYBRID_DENSE_K", str(max(TOP_K, 8))))
HYBRID_SPARSE_K = int(os.getenv("HYBRID_SPARSE_K", str(max(TOP_K, 8))))
HYBRID_RRF_K = int(os.getenv("HYBRID_RRF_K", "60"))
HYBRID_ALPHA = float(os.getenv("HYBRID_ALPHA", "0.6"))

EVAL_CSV_PATH = os.getenv("EVAL_CSV_PATH", "./eval_dataset.csv")
EVAL_OUT_CSV_PATH = os.getenv("EVAL_OUT_CSV_PATH", "./eval_results.csv")
EVAL_QUESTION_COL = os.getenv("EVAL_QUESTION_COL", "question")
EVAL_REFERENCE_COL = os.getenv("EVAL_REFERENCE_COL", "reference_answer")
EVAL_MAX_ROWS = int(os.getenv("EVAL_MAX_ROWS", "0"))  # 0 => all rows


# ----------------------------
# Env checks + Redis checks
# ----------------------------
def ensure_env():
    if not OPENAI_API_KEY:
        raise RuntimeError("Missing OPENAI_API_KEY (set it in .env).")


def assert_redis_up(redis_url: str):
    r = redis.from_url(redis_url)
    try:
        ok = r.ping()
        if ok is not True:
            raise RuntimeError("Redis PING failed.")
    except Exception as e:
        raise RuntimeError(f"Cannot connect to Redis at {redis_url}. Error: {repr(e)}")


def enable_langchain_cache(redis_url: str, ttl_seconds: int):
    try:
        try:
            langchain.llm_cache = RedisCache(redis_url=redis_url, ttl=ttl_seconds)
        except TypeError:
            langchain.llm_cache = RedisCache(redis_url=redis_url)
    except Exception as e:
        raise RuntimeError(f"Failed to enable Redis cache: {repr(e)}")


def clear_langchain_redis_cache(redis_url: str):
    r = redis.from_url(redis_url)
    keys = r.keys("langchain:llm_cache*")
    if keys:
        r.delete(*keys)
    print(f"Cleared {len(keys)} LangChain cache keys from Redis.")


# ----------------------------
# PDF loading + chunking
# ----------------------------
def find_pdfs(data_dir: str) -> List[str]:
    pattern = os.path.join(data_dir, "**", "*.pdf")
    return sorted(glob.glob(pattern, recursive=True))


def load_pdf_docs(pdf_paths: List[str], base_dir: str) -> List[Document]:
    docs: List[Document] = []
    for path in pdf_paths:
        loader = PyPDFLoader(path)
        loaded = loader.load()
        rel = os.path.relpath(path, base_dir)

        for d in loaded:
            d.metadata = d.metadata or {}
            d.metadata["source"] = rel
            docs.append(d)
    return docs


def chunk_docs(docs: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    return splitter.split_documents(docs)


def format_docs_with_sources(docs: List[Document]) -> str:
    out = []
    for i, d in enumerate(docs, start=1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        page_str = f", page {page}" if page is not None else ""
        out.append(f"---\nChunk {i} [source: {src}{page_str}]\n{d.page_content}")
    return "\n".join(out)


def extract_sources(docs: List[Document]) -> List[str]:
    s: Set[str] = set()
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        if page is not None:
            s.add(f"{src} (page {page})")
        else:
            s.add(src)
    return sorted(s)


# ----------------------------
# Build corpus + FAISS + BM25
# ----------------------------
def build_corpus_and_faiss() -> Tuple[List[Document], FAISS]:
    pdfs = find_pdfs(DATA_DIR)
    if not pdfs:
        raise RuntimeError(f"No PDFs found in {DATA_DIR}. Add PDFs and re-run.")

    raw_docs = load_pdf_docs(pdfs, base_dir=DATA_DIR)
    chunks = chunk_docs(raw_docs)

    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
    vs = FAISS.from_documents(chunks, embeddings)

    print(f"Ingested {len(chunks)} chunks from {len(pdfs)} PDFs into FAISS.")
    return chunks, vs


def simple_tokenize(text: str) -> List[str]:
    return [t for t in "".join(ch if ch.isalnum() else " " for ch in text.lower()).split() if t]


def build_bm25_index(docs: List[Document]) -> BM25Okapi:
    tokenized = [simple_tokenize(d.page_content) for d in docs]
    return BM25Okapi(tokenized)


# ----------------------------
# Hybrid retrieval (Dense + Sparse + RRF)
# ----------------------------
def rrf_fuse(
    dense_docs: List[Document],
    sparse_docs: List[Document],
    alpha: float,
    rrf_k: int,
    top_k: int,
) -> List[Document]:
    scores: Dict[str, float] = {}
    doc_by_id: Dict[str, Document] = {}

    def doc_id(d: Document) -> str:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", -1)
        prefix = (d.page_content[:200] if d.page_content else "")
        return f"{src}::{page}::{hash(prefix)}"

    for rank, d in enumerate(dense_docs, start=1):
        did = doc_id(d)
        doc_by_id[did] = d
        scores[did] = scores.get(did, 0.0) + alpha * (1.0 / (rrf_k + rank))

    for rank, d in enumerate(sparse_docs, start=1):
        did = doc_id(d)
        doc_by_id[did] = d
        scores[did] = scores.get(did, 0.0) + (1.0 - alpha) * (1.0 / (rrf_k + rank))

    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [doc_by_id[did] for did, _ in ranked[:top_k]]


class HybridRetriever:
    def __init__(self, faiss_store: FAISS, bm25: BM25Okapi, corpus_docs: List[Document]):
        self.dense_retriever = faiss_store.as_retriever(search_kwargs={"k": HYBRID_DENSE_K})
        self.bm25 = bm25
        self.corpus_docs = corpus_docs

    def sparse_retrieve(self, query: str, k: int) -> List[Document]:
        q_tokens = simple_tokenize(query)
        if not q_tokens:
            return []
        scores = self.bm25.get_scores(q_tokens)
        top_idx = np.argsort(scores)[::-1][:k]
        return [self.corpus_docs[int(i)] for i in top_idx]

    def invoke(self, query: str) -> List[Document]:
        dense_docs = self.dense_retriever.invoke(query)
        sparse_docs = self.sparse_retrieve(query, HYBRID_SPARSE_K)
        return rrf_fuse(dense_docs, sparse_docs, HYBRID_ALPHA, HYBRID_RRF_K, TOP_K)


# ----------------------------
# RAG Chains (Normal vs Hybrid)
# ----------------------------
def make_prompt() -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful assistant. Answer ONLY using the provided context. "
                "If the answer is not in the context, say you don't know. "
                "At the end include a short 'Sources:' list."
            ),
            ("human", "Question:\n{question}\n\nContext:\n{context}"),
        ]
    )


def build_rag_chain_normal(faiss_store: FAISS):
    retriever = faiss_store.as_retriever(search_kwargs={"k": TOP_K})
    prompt = make_prompt()
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

    rag_chain = (
        {
            "question": RunnablePassthrough(),
            "context": retriever | RunnableLambda(format_docs_with_sources),
        }
        | prompt
        | llm
    )

    def get_sources(q: str) -> Dict[str, Any]:
        docs = retriever.invoke(q)
        return {"sources": extract_sources(docs)}

    sources_chain = RunnableLambda(get_sources)
    return rag_chain, sources_chain


def build_rag_chain_hybrid(hybrid_retriever: HybridRetriever):
    prompt = make_prompt()
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

    rag_chain = (
        {
            "question": RunnablePassthrough(),
            "context": RunnableLambda(lambda q: hybrid_retriever.invoke(q))
            | RunnableLambda(format_docs_with_sources),
        }
        | prompt
        | llm
    )

    def get_sources(q: str) -> Dict[str, Any]:
        docs = hybrid_retriever.invoke(q)
        return {"sources": extract_sources(docs)}

    sources_chain = RunnableLambda(get_sources)
    return rag_chain, sources_chain


# ----------------------------
# Evaluation helpers
# ----------------------------
def safe_text(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x).strip()


def cosine_sim(vec_a: List[float], vec_b: List[float]) -> float:
    a = np.array(vec_a, dtype=np.float32)
    b = np.array(vec_b, dtype=np.float32)
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


# ----------------------------
# Main: 1st invoke normal, 2nd invoke hybrid
# ----------------------------
def main():
    ensure_env()
    assert_redis_up(REDIS_URL)
    enable_langchain_cache(REDIS_URL, REDIS_CACHE_TTL_SECONDS)
    clear_langchain_redis_cache(REDIS_URL)

    # Build corpus + FAISS + BM25
    corpus_docs, faiss_vs = build_corpus_and_faiss()
    bm25 = build_bm25_index(corpus_docs)
    hybrid_retriever = HybridRetriever(faiss_store=faiss_vs, bm25=bm25, corpus_docs=corpus_docs)

    # Build both chains
    rag_chain_normal, sources_normal = build_rag_chain_normal(faiss_vs)
    rag_chain_hybrid, sources_hybrid = build_rag_chain_hybrid(hybrid_retriever)

    # Load evaluation CSV
    if not os.path.exists(EVAL_CSV_PATH):
        raise FileNotFoundError(f"CSV not found: {EVAL_CSV_PATH}")

    df = pd.read_csv(EVAL_CSV_PATH)
    if EVAL_QUESTION_COL not in df.columns or EVAL_REFERENCE_COL not in df.columns:
        raise ValueError(
            f"CSV must contain columns '{EVAL_QUESTION_COL}' and '{EVAL_REFERENCE_COL}'. "
            f"Found: {list(df.columns)}"
        )

    if EVAL_MAX_ROWS and EVAL_MAX_ROWS > 0:
        df = df.head(EVAL_MAX_ROWS).copy()

    emb = OpenAIEmbeddings(model=EMBEDDING_MODEL)

    results = []
    sims_normal = []
    sims_hybrid = []
    lat1_list = []
    lat2_list = []

    print(f"Evaluating {len(df)} rows from {EVAL_CSV_PATH} ...")
    print("Invoke #1: NORMAL (dense FAISS only)")
    print("Invoke #2: HYBRID (FAISS + BM25 + RRF)")
    print(f"Hybrid params: dense_k={HYBRID_DENSE_K}, sparse_k={HYBRID_SPARSE_K}, alpha={HYBRID_ALPHA}, rrf_k={HYBRID_RRF_K}")

    for i, row in df.iterrows():
        question = safe_text(row[EVAL_QUESTION_COL])
        ref_answer = safe_text(row[EVAL_REFERENCE_COL])

        if not question:
            results.append(
                {
                    "row": int(i),
                    "question": question,
                    "reference_answer": ref_answer,
                    "answer_normal": "",
                    "answer_hybrid": "",
                    "cosine_normal": np.nan,
                    "cosine_hybrid": np.nan,
                    "latency_ms_normal": np.nan,
                    "latency_ms_hybrid": np.nan,
                    "sources_normal": "",
                    "sources_hybrid": "",
                    "error": "empty_question",
                }
            )
            continue

        try:
            # --- NORMAL sources + invoke ---
            src_norm = sources_normal.invoke(question).get("sources", [])
            src_norm_str = "; ".join(src_norm) if isinstance(src_norm, list) else safe_text(src_norm)

            t0 = time.perf_counter()
            ans1 = rag_chain_normal.invoke(question)
            t1 = time.perf_counter()
            lat_norm = (t1 - t0) * 1000.0
            answer_normal = safe_text(getattr(ans1, "content", str(ans1)))

            # --- HYBRID sources + invoke ---
            src_hyb = sources_hybrid.invoke(question).get("sources", [])
            src_hyb_str = "; ".join(src_hyb) if isinstance(src_hyb, list) else safe_text(src_hyb)

            t2 = time.perf_counter()
            ans2 = rag_chain_hybrid.invoke(question)
            t3 = time.perf_counter()
            lat_hyb = (t3 - t2) * 1000.0
            answer_hybrid = safe_text(getattr(ans2, "content", str(ans2)))

            # Similarities
            if ref_answer and answer_normal:
                cos_norm = cosine_sim(emb.embed_query(ref_answer), emb.embed_query(answer_normal))
            else:
                cos_norm = np.nan

            if ref_answer and answer_hybrid:
                cos_hyb = cosine_sim(emb.embed_query(ref_answer), emb.embed_query(answer_hybrid))
            else:
                cos_hyb = np.nan

            if not np.isnan(cos_norm):
                sims_normal.append(cos_norm)
            if not np.isnan(cos_hyb):
                sims_hybrid.append(cos_hyb)

            lat1_list.append(lat_norm)
            lat2_list.append(lat_hyb)

            results.append(
                {
                    "row": int(i),
                    "question": question,
                    "reference_answer": ref_answer,
                    "answer_normal": answer_normal,
                    "answer_hybrid": answer_hybrid,
                    "cosine_normal": cos_norm,
                    "cosine_hybrid": cos_hyb,
                    "latency_ms_normal": lat_norm,
                    "latency_ms_hybrid": lat_hyb,
                    "sources_normal": src_norm_str,
                    "sources_hybrid": src_hyb_str,
                    "error": "",
                }
            )

            if (len(results) % 10) == 0:
                avg_n = float(np.mean(sims_normal)) if sims_normal else float("nan")
                avg_h = float(np.mean(sims_hybrid)) if sims_hybrid else float("nan")
                p50_n = float(np.median(lat1_list)) if lat1_list else float("nan")
                p50_h = float(np.median(lat2_list)) if lat2_list else float("nan")
                print(
                    f"Processed {len(results)}/{len(df)} | "
                    f"avg cosine normal={avg_n:.4f}, hybrid={avg_h:.4f} | "
                    f"p50 latency normal={p50_n:.1f} ms, hybrid={p50_h:.1f} ms"
                )

        except Exception as e:
            results.append(
                {
                    "row": int(i),
                    "question": question,
                    "reference_answer": ref_answer,
                    "answer_normal": "",
                    "answer_hybrid": "",
                    "cosine_normal": np.nan,
                    "cosine_hybrid": np.nan,
                    "latency_ms_normal": np.nan,
                    "latency_ms_hybrid": np.nan,
                    "sources_normal": "",
                    "sources_hybrid": "",
                    "error": repr(e),
                }
            )

    out_df = pd.DataFrame(results)
    out_df.to_csv(EVAL_OUT_CSV_PATH, index=False)

    print("\n=== Evaluation Summary ===")
    print(f"Rows processed: {len(out_df)}")

    if sims_normal:
        print(f"Cosine NORMAL mean={float(np.mean(sims_normal)):.4f}, median={float(np.median(sims_normal)):.4f}")
    else:
        print("Cosine NORMAL: n/a")

    if sims_hybrid:
        print(f"Cosine HYBRID mean={float(np.mean(sims_hybrid)):.4f}, median={float(np.median(sims_hybrid)):.4f}")
    else:
        print("Cosine HYBRID: n/a")

    if lat1_list:
        print(f"Latency NORMAL mean={float(np.mean(lat1_list)):.1f} ms | p50={float(np.median(lat1_list)):.1f} ms")
    else:
        print("Latency NORMAL: n/a")

    if lat2_list:
        print(f"Latency HYBRID mean={float(np.mean(lat2_list)):.1f} ms | p50={float(np.median(lat2_list)):.1f} ms")
    else:
        print("Latency HYBRID: n/a")

    print(f"Saved results to: {EVAL_OUT_CSV_PATH}")


if __name__ == "__main__":
    main()


Cleared 0 LangChain cache keys from Redis.
Ingested 70 chunks from 1 PDFs into FAISS.
Evaluating 50 rows from ./RA_FSM_QA.csv ...
Invoke #1: NORMAL (dense FAISS only)
Invoke #2: HYBRID (FAISS + BM25 + RRF)
Hybrid params: dense_k=8, sparse_k=8, alpha=0.6, rrf_k=60
Processed 10/50 | avg cosine normal=0.5204, hybrid=0.5152 | p50 latency normal=2954.7 ms, hybrid=2656.0 ms
Processed 20/50 | avg cosine normal=0.5098, hybrid=0.5107 | p50 latency normal=2405.5 ms, hybrid=2126.8 ms
Processed 30/50 | avg cosine normal=0.5107, hybrid=0.5140 | p50 latency normal=2267.5 ms, hybrid=2165.7 ms
Processed 40/50 | avg cosine normal=0.4838, hybrid=0.4937 | p50 latency normal=2267.5 ms, hybrid=2259.2 ms
Processed 50/50 | avg cosine normal=0.4823, hybrid=0.4969 | p50 latency normal=2370.5 ms, hybrid=2229.4 ms

=== Evaluation Summary ===
Rows processed: 50
Cosine NORMAL mean=0.4823, median=0.4941
Cosine HYBRID mean=0.4969, median=0.5009
Latency NORMAL mean=2517.3 ms | p50=2370.5 ms
Latency HYBRID mean=2435.0